The goal is to get an overall sense of how privacy-sensitive a database is and which attributes make it sensitive.  

Follows directly the method in :
S.-H. Kim, C. Jung, and Y.-J. Lee, “An entropy-based analytic model for the privacy-preserving in open data,” in 2016 IEEE International Conference on Big Data (Big Data), Dec. 2016, pp. 3676–3684. doi: 10.1109/BigData.2016.7841035.


Find out how likely it is that the attacker can match all the quasi identifiers to get the sensitive attribute:

i_entity: is the information that attacker has, basically everything except the sensitive attribute

l_entity: the database we want to release with the sensitive data.  


In [ ]:
import math

## 1. Load Dataset

In [ ]:
import pandas as pd

df = pd.read_csv("synthetic_dataset.csv")

for col in df.select_dtypes(include="object").columns:
    df[col] = df[col].astype(str)

print(f"Loaded dataset: {df.shape[0]} rows × {df.shape[1]} columns")
df.head()


## 2. Column Classification

In [ ]:
all_columns = list(df.columns)
print("Columns:", all_columns)

column_classification = {
    "semi_identifiers": ['url','language', 'country', 'device', 'star_review'],  #  fill in
    "sensitive":        ['text_message'],  # fill in
}

print("Semi-identifiers:", column_classification["semi_identifiers"])
print("Sensitive:       ", column_classification["sensitive"])


In [ ]:
all_cols = column_classification["semi_identifiers"] + column_classification["sensitive"]
df = df[all_cols]


I-Entity Dataset

The I-entity dataset is formed by dropping the sensitive attribute(s), retaining only identifiers and semi-identifiers.

In [ ]:
# TODO  For a actual test, use a different dataset, but this is what the paper does.
df_l_entity = df # The database we will release with the sensitive data
df_i_entity = df[column_classification["semi_identifiers"]].copy()  # what the attacker knows, with only the semi_identifiers

print(f"Original dataset : {df.shape}")
print(f"I-entity dataset : {df_i_entity.shape}")
print(f"Dropped columns  : {column_classification['sensitive']}")
df_i_entity.head()

## 4. Matching Probabilities

Group the S-entity dataset by quasi-identifiers (semi-identifiers) to form **equivalence classes** — sets of records that are indistinguishable on those attributes.

For a target I-entity, the **matching probability** is the chance an attacker correctly links one record in its equivalence class to that individual:

$$P_{match} = \frac{1}{|equivalence\ class|}$$

A smaller class → higher matching probability → greater re-identification risk.

In [ ]:
# Data exploration only, this is also done again in the matching_probabilities function.

quasi_ids = column_classification["semi_identifiers"]

# Size of each equivalence class
equiv_classes = (
    df_i_entity
    .groupby(quasi_ids)
    .size()
    .reset_index(name="class_size")
)
equiv_classes["match_probability"] = 1 / equiv_classes["class_size"]

equiv_classes = equiv_classes.sort_values("class_size").reset_index(drop=True)

print(f"Total equivalence classes : {len(equiv_classes)}")
print(f"Smallest class size       : {equiv_classes['class_size'].min()}")
print(f"Largest class size        : {equiv_classes['class_size'].max()}")
print(f"Mean class size           : {equiv_classes['class_size'].mean():.2f}")
equiv_classes.head(10)

In [ ]:
ququasi_ids = column_classification["semi_identifiers"]

rows = []
baseline_classes = df_i_entity.groupby(quasi_ids).ngroups

for col in quasi_ids:
    reduced = [q for q in quasi_ids if q != col]
    n_classes = df_i_entity.groupby(reduced).ngroups if reduced else len(df_i_entity)
    rows.append({
        "column":           col,
        "unique_values":    df_i_entity[col].nunique(),
        "equiv_classes_without": n_classes,
    })

summary_df = pd.DataFrame(rows)
summary_df["equiv_classes_baseline"] = baseline_classes
summary_df["class_reduction"] = baseline_classes - summary_df["equiv_classes_without"]

print(f"Baseline equivalence classes (all semi-identifiers): {baseline_classes}\n")
print(summary_df.to_string(index=False))


In [ ]:
cols_to_remove = ['star_review', 'device']

column_classification["semi_identifiers"] = [
    c for c in column_classification["semi_identifiers"] if c not in cols_to_remove
]
df_l_entity = df_l_entity.drop(columns=cols_to_remove, errors="ignore")
df_i_entity = df_i_entity.drop(columns=cols_to_remove, errors="ignore")

print("Updated semi-identifiers:", column_classification["semi_identifiers"])


In [ ]:
def probability_bin_counts(probabilities: pd.Series, bins: int = 10) -> pd.Series:
    """Return counts of probability values in equal-width bins across [0, 1]."""
    bin_edges = [i / bins for i in range(bins + 1)]
    bin_labels = [f"{i/bins:.1f}-{(i+1)/bins:.1f}" for i in range(bins)]

    binned = pd.cut(
        probabilities.dropna(),
        bins=bin_edges,
        labels=bin_labels,
        include_lowest=True,
        right=True,
    )
    return binned.value_counts(sort=False)

def matching_probabilities(
    s_df: pd.DataFrame,
    i_df: pd.DataFrame,
    quasi_ids: list[str],
    sensitive_col: str,
) -> pd.DataFrame:
    """Calculate P(s|i) for every I-entity / sensitive-value pair.

    For each individual i in i_df, finds all records in s_df sharing the same
    quasi-identifier values (the matched group), then computes the probability
    of each sensitive value within that group.

    Args:
        s_df:          S-entity dataset (open data) — quasi-identifiers + sensitive attribute.
        i_df:          I-entity dataset (external data) — quasi-identifiers only.
        quasi_ids:     Quasi-identifier column names to match on.
        sensitive_col: Sensitive attribute column name in s_df.

    Returns:
        DataFrame with one row per (i_entity, sensitive_value) pair and columns:
            i_entity_index  — original row index from i_df
            <quasi_id cols> — quasi-identifier values for that i-entity
            <sensitive_col> — a distinct sensitive value found in the matched group
            match_count     — occurrences of that sensitive value in the group
            class_size      — total size of the matched group\n
            probability     — match_count / class_size  (p(s|i))
        Unmatched i-entities are included with NaN probability.
    """
   # Pre-compute frequency distribution for every equivalence class in s_df
    group_counts = (
        s_df.groupby(quasi_ids + [sensitive_col])
        .size()
        .reset_index(name="match_count")
    )
    equiv_classes = (
        s_df.groupby(quasi_ids)
        .size()
        .reset_index(name="class_size")
    )
    group_dist = group_counts.merge(equiv_classes, on=quasi_ids)
    group_dist["probability"] = group_dist["match_count"] / group_dist["class_size"]

    # Tag each I-entity with its original index, then merge on quasi-identifiers
    i_tagged = i_df[quasi_ids].copy()
    i_tagged.insert(0, "i_entity_index", i_tagged.index)


    result = i_tagged.merge(group_dist, on=quasi_ids, how="left")
    #result = result.rename(columns={sensitive_col: "sensitive_value"})
    
    print("Probability bin counts (10 bins, 0-1):")
    print(probability_bin_counts(result["probability"], bins=10).to_string())

    return result.reset_index(drop=True)


# --- Example call ---
sensitive_col = column_classification["sensitive"][0]

prob_df = matching_probabilities(df_l_entity, df_i_entity, column_classification["semi_identifiers"], sensitive_col)
print(f"Result shape: {prob_df.shape}")
print(prob_df.head(20).to_string(index=False))


## 5. Entropy Measures per I-Entity

Three measures are computed for each I-entity $I_j$ from its sensitive attribute probability distribution:

| Measure | Formula | Interpretation |
|---|---|---|
| **Matching entropy** $H(S\|I_j)$ | $-\sum_i P(S_i, I_j)\log_2 P(S_i, I_j)$ | Uncertainty of linking $I_j$ to a sensitive value |
| **Maximum entropy** $H_{max}$ | $\log_2(n)$ where $n$ = unique sensitive values | Upper bound — perfectly uniform distribution |
| **Relative entropy** $D$ | $H_{max} - H(S\|I_j)$ | How far $I_j$'s distribution is from maximum uncertainty; higher = more re-identification risk |

In [ ]:
def entropy_measures(prob_df: pd.DataFrame, sensitive_col: str) -> pd.DataFrame:
    """Calculate H, H_max, and D for each I-entity from matching_probabilities output.

    Args:
        prob_df: Output of matching_probabilities() — one row per (I_j, S_i) pair
                 with columns i_entity_index, <sensitive_col>, probability.

    Returns:
        DataFrame with one row per I-entity and columns:
            i_entity_index  — original index from i_df
            H               — matching entropy H(S|I_j) in bits
            H_max           — log2(n_unique_sensitive_values)
            D               — relative entropy H_max - H
    """
    n_sensitive = prob_df[sensitive_col].nunique()
    h_max = math.log2(n_sensitive) if n_sensitive > 1 else 0.0

    def _h(group):
        p = group["probability"].dropna()
        # Only sum terms where p > 0 (log2(0) is undefined)
        return -(p[p > 0] * p[p > 0].apply(math.log2)).sum()

    matching_entropy = (
        prob_df.groupby("i_entity_index")
        .apply(_h, include_groups=False)
        .reset_index(name="H")
    )

    matching_entropy["H_max"] = h_max
    matching_entropy["D"] = matching_entropy["H_max"] - matching_entropy["H"]

    return matching_entropy


if "prob_df" in globals():
    entropy_df = entropy_measures(prob_df, sensitive_col)

    print(f"H_max (global)  : {entropy_df['H_max'].iloc[0]:.4f} bits")
    print(f"Mean H          : {entropy_df['H'].mean():.4f} bits")
    print(f"Mean D          : {entropy_df['D'].mean():.4f} bits")
    print(f"\nHighest-risk I-entities (lowest H, highest D):")
    print(entropy_df.sort_values("D", ascending=False).head(10).to_string(index=False))
else:
    print("Run the matching_probabilities cell first to produce prob_df.")

## 6. Re-identification Risk

- **Entity risk** — how close each I-entity's distribution is to being fully determined: $Risk_j = (D_j / H_{max}) \times 100$
- **Dataset risk** — the maximum entity risk, representing the weakest point in the dataset

In [ ]:
entropy_df["risk_pct"] = (entropy_df["D"] / entropy_df["H_max"]) * 100

dataset_risk = entropy_df["risk_pct"].max()
worst = entropy_df.loc[entropy_df["risk_pct"].idxmax()]

print(f"Dataset re-identification risk : {dataset_risk:.2f}%")
print(f"Most vulnerable I-entity index : {int(worst['i_entity_index'])}")
print(f"  H      = {worst['H']:.4f} bits")
print(f"  H_max  = {worst['H_max']:.4f} bits")
print(f"  D      = {worst['D']:.4f} bits")
print(f"  Risk   = {worst['risk_pct']:.2f}%")

print("\nEntity risk distribution:")
print(entropy_df["risk_pct"].describe().round(2))

## 7. Quasi-Identifier Sensitivity Analysis

Remove one quasi-identifier at a time and recompute entropy measures. This shows which column contributes most to re-identification risk — removing it causes the largest drop in dataset risk.

In [ ]:
baseline_risk = dataset_risk  # already computed in section 6

sensitivity_records = []

for col in quasi_ids:
    reduced_quasi_ids = [q for q in quasi_ids if q != col]

    reduced_prob_df = matching_probabilities(df, df_i_entity, reduced_quasi_ids, sensitive_col)
    reduced_entropy_df = entropy_measures(reduced_prob_df, sensitive_col)
    reduced_entropy_df["risk_pct"] = (reduced_entropy_df["D"] / reduced_entropy_df["H_max"]) * 100

    sensitivity_records.append({
        "removed_column":  col,
        "mean_H":          reduced_entropy_df["H"].mean(),
        "mean_D":          reduced_entropy_df["D"].mean(),
        "dataset_risk_pct": reduced_entropy_df["risk_pct"].max(),
        "risk_delta":      reduced_entropy_df["risk_pct"].max() - baseline_risk,
    })

sensitivity_df = (
    pd.DataFrame(sensitivity_records)
    .sort_values("risk_delta")          # most negative = biggest risk reduction
    .reset_index(drop=True)
)

print(f"Baseline dataset risk : {baseline_risk:.2f}%\n")
print(sensitivity_df.round(4).to_string(index=False))

The attack is:  You know there is only one person who would search for roblox in a given location.  You find them in the database based on the quasi-identifiers and sensitive value.
Now, can you match the quasi-identifiers to get other sensitive values. 
